In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing import TypedDict, Annotated
from dotenv import load_dotenv
import os, requests

In [10]:
# os.environ['LANGCHAIN_PROJECT'] = 'langchain_demo'

In [2]:
load_dotenv()

llm = ChatOpenAI()

In [11]:
search_tool = DuckDuckGoSearchRun(region="us-en")

@tool
def calculator_tool(a:float, b:float, operation:str)->dict:
    """Perform a basic arithmetic operation on two numbers.
    Suported operations - add, sub, mul, div"""

    try:
        if operation == "add":
            result = a+b
        elif operation == "sub":
            result = a-b
        elif operation == "mul":
            result = a*b            
        elif operation == "div":
            if b==0:
                return {"error":"Division by zero is not allowed"}
            result = a/b
        else:
            return {"error":f"Unsupported opertation {operation}"}
        return {"first_num":a,"second_num":b,"operation":operation,"result":result}
    except Exception as e:
        return {"error":str(e)}
    
@tool
def stock_tool(stock:str)->dict:
    """Fetch latest stock for a given symbol (eg 'AAPL', 'TSLA')
    using Alpha vantage with API key in the URL"""
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={stock}&apikey=9TH1EV1XBL8FTQ8J"
    response = requests.get(url)
    return response.json()

In [13]:
tools = [search_tool, calculator_tool, stock_tool]

toolnode = ToolNode(tools)

llm_tools = llm.bind_tools(tools)

In [3]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [14]:
def chat_node(state:ChatState):
    """LLM node that may answer or request a tool call"""
    response = llm_tools.invoke(state['messages'])
    return {'messages':response}

In [19]:
graph = StateGraph(ChatState)

graph.add_node("chat_node",chat_node)
graph.add_node("tools",toolnode)

graph.add_edge(START,"chat_node")
graph.add_conditional_edges("chat_node",tools_condition)
graph.add_edge("tools","chat_node")

chatbot = graph.compile()

In [22]:
out = chatbot.invoke({"messages":[HumanMessage(content="Hello")]})

print(out["messages"][-1].content)

Hello! How can I assist you today?


In [23]:
out = chatbot.invoke({"messages":[HumanMessage(content="Who owns YouTube")]})

print(out["messages"][-1].content)

YouTube is owned by the Google parent company, Alphabet. Sergey Brin and Larry Page control a majority stake in Alphabet, while Vanguard and BlackRock are the two largest institutional investors. With Google acquiring YouTube in 2006 for $1.65 billion, it became a wholly-owned subsidiary of Google, which is now under the umbrella of Alphabet Inc.


In [25]:
out = chatbot.invoke({"messages":[HumanMessage(content="What is 300 multipled by 4 which is then divided by 2")]})

print(out["messages"][-1].content)

The result of multiplying 300 by 4 and then dividing by 2 is 600.


In [26]:
out = chatbot.invoke({"messages":[HumanMessage(content="What is the current stock price of Apple")]})

print(out["messages"][-1].content)

The current stock price of Apple (AAPL) is $259.48.


In [28]:
out = chatbot.invoke({"messages": [HumanMessage(content="First find out the stock price of Apple then use that to find out how much will it take to purchase 50 shares? Also find out who owns apple")]})
print(out["messages"][-1].content)

The current stock price of Apple Inc. (AAPL) is $259.48 per share. 

To purchase 50 shares of Apple, it would cost $12,974.00.

As for the ownership of Apple Inc., it is diversified among several corporations and individuals.
